In [1]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA L4


In [29]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [30]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece liger-kernel
!pip uninstall -y torchao -q

In [2]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece
!pip uninstall -y torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import torch
import gc

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

set_seed(42)

In [6]:
# Full dataset train onek shomoy nibe, tai checkpoint Google Drive-e rakhte mount kora hocche
# (Colab session disconnect hoyeo train.resume_from_checkpoint diye continue kora jabe)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
OUTPUT_DIR = "/content/drive/MyDrive/qwen_medical_bengali"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive


## Data

In [7]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [8]:
df = pd.read_csv("train.csv")
df = df[["id", "input", "output"]]

# ekhon ful dataset use hocche - kono sampling nai
df = df.dropna(subset=["input", "output"]).reset_index(drop=True)
print(df.shape)

train_df, val_df = train_test_split(df, test_size=0.05, random_state=42)
print(train_df.shape, val_df.shape)

(108954, 3)
(103506, 3) (5448, 3)


## Tokenizer

In [9]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocabulary Size:", tokenizer.vocab_size)
print("EOS Token:", tokenizer.eos_token)
print("PAD Token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Vocabulary Size: 151643
EOS Token: <|im_end|>
PAD Token: <|endoftext|>


In [10]:
SYSTEM_PROMPT = (
    "আপনি একজন অভিজ্ঞ বাংলা চিকিৎসক। "
    "রোগীর প্রশ্নে যদি পর্যাপ্ত তথ্য না থাকে (যেমন: লক্ষণের সময়কাল, তীব্রতা, অবস্থান, সহগামী উপসর্গ), তাহলে সরাসরি উত্তর দেওয়ার আগে প্রাসঙ্গিক স্পষ্টীকরণ প্রশ্ন জিজ্ঞাসা করুন —যেমন 'আপনার ব্যথা কতদিন ধরে হচ্ছে?', 'ব্যথার সাথে জ্বর বা বমি ভাব আছে কি?',ব্যথাটা পেটের কোন অংশে বেশি অনুভব করছেন?' ইত্যাদি। "
    "একসাথে সব প্রশ্ন না করে, সবচেয়ে গুরুত্বপূর্ণ ২-৩টি প্রশ্ন জিজ্ঞাসা করুন। "
    "পর্যাপ্ত তথ্য পাওয়ার পরই সম্ভাব্য কারণ ব্যাখ্যা করুন এবং পরামর্শ দিন। "
    "অপ্রয়োজনীয় তথ্য, পুনরাবৃত্তি বা অনুমানভিত্তিক দাবি করবেন না। "
    "উত্তর হবে স্বাভাবিক, আশ্বস্তকারী, পেশাদার এবং রোগী-বান্ধব বাংলায়।"
    "আপনার টোন হবে স্বাভাবিক, পেশাদার এবং রোগী-বান্ধব। "
    "সাধারণ বা মৃদু উপসর্গের ক্ষেত্রে আশ্বস্তকারী হোন, কিন্তু গুরুতর বা জরুরি লক্ষণের ক্ষেত্রে (যেমন: তীব্র বুকে ব্যথা, শ্বাসকষ্ট, অতিরিক্ত রক্তক্ষরণ, অজ্ঞান হয়ে যাওয়া, উচ্চ জ্বরসহ ঘাড় শক্ত হওয়া) আশ্বস্ত করার চেষ্টা না করে স্পষ্টভাবে জরুরি চিকিৎসা সেবা নিতে বলুন।"
)
MAX_LENGTH = 2048

## Format + Tokenize + Label masking (single pass)

Assistant-er response tokens-er upor loss lagano hocche (prompt part -100 diye mask kora).

In [11]:
def format_and_tokenize(example):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
    ]
    full_messages = prompt_messages + [
        {"role": "assistant", "content": example["output"]}
    ]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        full_messages, tokenize=False, add_generation_prompt=False
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(
        full_text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False
    )["input_ids"]

    prompt_len = min(len(prompt_ids), len(full_ids))

    labels = full_ids.copy()
    for i in range(prompt_len):
        labels[i] = -100

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

In [12]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(format_and_tokenize, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(format_and_tokenize, remove_columns=val_dataset.column_names)

print(train_dataset)
print(train_dataset.column_names)

Map:   0%|          | 0/103506 [00:00<?, ? examples/s]

Map:   0%|          | 0/5448 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 103506
})
['input_ids', 'attention_mask', 'labels']


In [13]:
def has_supervised_tokens(example):
    return any(l != -100 for l in example["labels"])

before = len(train_dataset)
train_dataset = train_dataset.filter(has_supervised_tokens)
val_dataset = val_dataset.filter(has_supervised_tokens)
after = len(train_dataset)

print(f"Train: {before} -> {after} (dropped {before - after} fully-truncated examples)")

Filter:   0%|          | 0/103506 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5448 [00:00<?, ? examples/s]

Train: 103506 -> 100178 (dropped 3328 fully-truncated examples)


In [14]:
lengths = [len(x) for x in train_dataset["input_ids"]]
print("Shortest:", min(lengths))
print("Average :", sum(lengths) / len(lengths))
print("Longest :", max(lengths))

Shortest: 1006
Average : 1950.876719439398
Longest : 2048


## Data collator

Custom collator jeta labels-o pad kore -100 diye. `DataCollatorForLanguageModeling` use kora jabe na, oita label masking overwrite kore fele.

In [15]:
def custom_data_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)
    pad_id = tokenizer.pad_token_id

    input_ids, attention_mask, labels = [], [], []
    for f in features:
        n_pad = max_len - len(f["input_ids"])
        input_ids.append(f["input_ids"] + [pad_id] * n_pad)
        attention_mask.append(f["attention_mask"] + [0] * n_pad)
        labels.append(f["labels"] + [-100] * n_pad)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

data_collator = custom_data_collator

## Model (QLoRA)

In [ ]:
from liger_kernel.transformers import apply_liger_kernel_to_qwen2
apply_liger_kernel_to_qwen2()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

print(type(model))
print(model.device)

In [17]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## Training args

In [ ]:
import transformers
print("Transformers version:", transformers.__version__)

_common_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    learning_rate=2e-4,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_strategy="steps",
    logging_steps=20,
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    remove_unused_columns=False,
    seed=42,
)

try:
    training_args = TrainingArguments(eval_strategy="steps", **_common_kwargs)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="steps", **_common_kwargs)

print(training_args)

In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [27]:
print("Allocated:", round(torch.cuda.memory_allocated()/1024**3, 2), "GB")
print("Reserved :", round(torch.cuda.memory_reserved()/1024**3, 2), "GB")

Allocated: 12.63 GB
Reserved : 13.79 GB


In [ ]:
import os
last_ckpt = None
if os.path.isdir(OUTPUT_DIR):
    ckpts = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        last_ckpt = os.path.join(OUTPUT_DIR, sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1])
        print("Resuming from:", last_ckpt)

trainer.train(resume_from_checkpoint=last_ckpt)

## Save + cleanup

In [ ]:
FINAL_DIR = OUTPUT_DIR + "_final"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

del trainer, model
gc.collect()
torch.cuda.empty_cache()

print("Model saved to", FINAL_DIR)

## Inference (merged model - fast)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(base_model, FINAL_DIR)
model = model.merge_and_unload()
model.config.use_cache = True
model.eval()

print("Free GPU memory:", torch.cuda.mem_get_info()[0] / 1024**3, "GB")

In [ ]:
from tqdm.auto import tqdm

def build_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_batch(questions, batch_size=16):
    predictions = []
    pbar = tqdm(total=len(questions), desc="Generating")
    for i in range(0, len(questions), batch_size):
        batch = questions[i:i+batch_size]
        prompts = [build_prompt(q) for q in batch]

        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_LENGTH,
        ).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                repetition_penalty=1.3,
                no_repeat_ngram_size=3,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        input_len = inputs.input_ids.shape[1]
        for out in outputs:
            predictions.append(tokenizer.decode(out[input_len:], skip_special_tokens=True).strip())

        pbar.update(len(batch))
    pbar.close()
    return predictions

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
test_df = pd.read_csv("test.csv")
print(test_df.shape)
test_df.head()

In [ ]:
predictions = generate_batch(test_df["input"].tolist(), batch_size=16)

In [ ]:
submission = pd.DataFrame({"id": test_df["id"], "output": predictions})

# empty string / NaN thakle fallback bosao - Kaggle null accept kore na
submission["output"] = submission["output"].fillna("").astype(str).str.strip()
empty_count = (submission["output"] == "").sum()
print(f"Empty predictions found: {empty_count}")

submission.loc[submission["output"] == "", "output"] = "দুঃখিত, উত্তর তৈরি করা সম্ভব হয়নি।"

submission.to_csv("submission.csv", index=False, encoding="utf-8")
print("Submission Saved!")

In [ ]:
from google.colab import files
files.download("submission.csv")